# Logistic Regression

Predicts the probability for a given binary class.

**Inputs**

- $\mathbf{X} \in [B, D_{in}]$
- $\mathbf{y} \in [B, D_{out}]$, lets assume $D_{out} = 1$ with binary classification

**Model**

$$\mathbf{z} = \mathbf{X W} + b$$

$$f(\mathbf{z}) = \frac{1}{1+e^{-\mathbf{z}}}$$

**Loss**

Binary cross entropy

$$\mathbb{l} = -\frac{1}{B} \sum_i (y_i \log f(z_i) + (1-y_i) \log (1-f(z_i)))$$


**Gradient Derivation**

$$
\frac{d\mathbb{l}}{d\mathbf{W}} = \frac{\partial \mathbf{z}}{\partial \mathbf{W}}\frac{\partial f}{\partial \mathbf{z}}\frac{\partial l}{\partial f}  
$$

$$
\frac{d\mathbb{l}}{db} = \frac{\partial z}{\partial b}\frac{\partial f}{\partial z} \frac{\partial l}{\partial f} 
$$

Use the following derivatives:

$$\frac{\partial \mathbb{l}}{\partial f} = -\frac{1}{B}(\frac{\mathbf{y}}{f} - \frac{1-\mathbf{y}}{1-f})$$

$$\frac{\partial f}{\partial \mathbf{z}} = -\frac{1}{(1+e^{-\mathbf{z}})^2} (-e^{-\mathbf{z}}) \\
= \frac{e^{-\mathbf{z}}}{(1+e^{-\mathbf{z}})^2} \\
= f(1-f)
$$

$$
\frac{\partial \mathbf{z}}{\partial \mathbf{W}} = \mathbf{X}^T
$$

$$
\frac{\partial \mathbf{z}}{\partial b} = 1|_{(B \times 1)}
$$



we can finally reach:

$$
\frac{dl}{d \mathbf{W}} = \frac{1}{B} \mathbf{X}^T(f- \mathbf{y})
$$

$$
\frac{dl}{db} = \frac{1}{B}1|_{(1, B)}(f- \mathbf{y}) = \frac{1}{B}\sum_{i}^{N}(f_i-y_i)
$$

## Data and Settings

In [14]:
import torch 
import numpy as np

In [15]:
B, Di, Do = 100, 10, 1

X = np.random.randn(B, Di)
y = np.random.randn(B, Do)
y = (y >= 0.5).astype(np.float32)

max_iters = 1000
lr = 1e-2

## Numpy Implementation

In [16]:
def model(x, W, b):
    """
    Args:
        x: (B, Di)
        W: (Di, Do)
        b: (1,)
    Returns:
        (B, Do)
    """
    z = x @ W + b
    f = 1 / (1 + np.exp(-z))
    
    return f

def binary_cross_entropy(pred, target):
    """
    Args:
        pred: (B, Do)
        target: (B, Do)
    Returns:
        scalar
    """
    eps = 1e-8
    loss = - (target * np.log(pred + eps) + (1 - target) * np.log(1 - pred + eps))
    return np.mean(loss)

def gradients(pred, target, x):
    """
    Args:
        pred: (B, Do)
        target: (B, Do)
        x: (B, Di) 
    """
    B, _= pred.shape
    dw = 1/B * x.T @ (pred - target)
    db = 1/B * np.sum(pred - target)
    return dw, db


In [17]:
# TRAINING 
# initialize weights
W = np.random.randn(Di, Do)
b = np.random.randn(1)

for i in range(max_iters):
    # forward 
    pred = model(X, W, b)

    # loss 
    loss = binary_cross_entropy(pred, y)

    # backward for gradient computation
    dw, db = gradients(pred, y, X)

    # update weights
    W -= lr * dw
    b -= lr * db

    # log 
    if i % 100 == 0:
        print(f"iter {i}: loss {loss:.4f}")

iter 0: loss 1.7398
iter 100: loss 1.5057
iter 200: loss 1.3022
iter 300: loss 1.1287
iter 400: loss 0.9844
iter 500: loss 0.8680
iter 600: loss 0.7775
iter 700: loss 0.7098
iter 800: loss 0.6612
iter 900: loss 0.6275
